# LLMs & Prompt Engineering — Local Inference with Ollama

**Phase 02 · Notebook 1 of 2**

---

## ⚠️ Prerequisites — Do This Before Running the Notebook

This notebook communicates with a **locally-running** language model via
[Ollama](https://ollama.ai). No cloud accounts or API keys are needed.

**Step 1 — Install Ollama**
```bash
# macOS / Linux
curl -fsSL https://ollama.ai/install.sh | sh

# Windows: download the installer from https://ollama.ai/download
```

**Step 2 — Pull the model**
```bash
ollama pull llama3
# This downloads ~4.7 GB once and caches it locally.
# Subsequent runs are instant.
```

**Step 3 — Verify Ollama is running**
```bash
curl http://localhost:11434/api/tags
# Should return a JSON list that includes llama3.
```

Keep Ollama running in the background while you work through this notebook.

---

**By the end of this notebook you will be able to:**

- Call a local LLM via Python `requests` with no frameworks
- Write prompts using four professional patterns: direct, few-shot, chain-of-thought, persona
- Use few-shot prompting to get structured classification output
- Reliably extract JSON from an LLM and parse it into a DataFrame
- Build a stateful chatbot loop that remembers previous turns

**Tools used:** `requests`, `pandas`, `json`, `faker` (data generation), Ollama (`llama3`)

---

## Section 1 — Calling Ollama from Python

### What is Ollama?

Ollama is a lightweight runtime that lets you run open-weight language models
(Llama 3, Mistral, Gemma, Phi-3, …) entirely on your own machine. It exposes a
**local REST API** that mirrors the OpenAI chat completions interface, so
everything you learn here transfers directly to production OpenAI/Anthropic
calls with only a URL and key change.

### Why local LLMs matter

| | Cloud API (GPT-4o, Claude 3.5) | Local (Ollama + llama3) |
|---|---|---|
| **Privacy** | Data leaves your network | Stays on your machine |
| **Cost** | $0.005–0.015 per 1k tokens | Free after hardware |
| **Latency** | 200ms–2s round-trip | 50–500ms depending on GPU |
| **Availability** | Requires internet | Works offline |
| **Model quality** | State-of-the-art | Slightly behind, closing fast |

For development, learning, and privacy-sensitive workloads, local inference is
the right default.

### How the REST API works

Ollama exposes **`POST http://localhost:11434/api/chat`** with a JSON body:

```json
{
  "model": "llama3",
  "messages": [{"role": "user", "content": "Hello!"}],
  "stream": false
}
```

Setting `stream: false` waits for the full response before returning. Setting
`stream: true` returns newline-delimited JSON chunks as they are generated —
useful for building typewriter-effect UIs, but harder to parse in notebooks.

### What to look for
- The response has `message.role = "assistant"` and `message.content = "..."`
- `done: true` confirms the model has finished
- `total_duration` (nanoseconds) tells you the generation speed

In [ ]:
import requests
import json
import textwrap

OLLAMA_URL = 'http://localhost:11434/api/chat'
MODEL      = 'llama3'


def chat(messages: list[dict], model: str = MODEL, temperature: float = 0.7) -> str:
    """Send a messages list to Ollama and return the assistant reply string."""
    payload = {
        'model':   model,
        'messages': messages,
        'stream':  False,
        'options': {'temperature': temperature},
    }
    try:
        resp = requests.post(OLLAMA_URL, json=payload, timeout=120)
        resp.raise_for_status()
    except requests.exceptions.ConnectionError:
        return ('[ERROR] Cannot reach Ollama at localhost:11434.\n'
                'Make sure Ollama is running: run `ollama serve` in a terminal.')
    data = resp.json()
    return data['message']['content']


def pp(text: str, width: int = 90) -> None:
    """Pretty-print long text with word wrapping."""
    print(textwrap.fill(text, width=width))


# ── First call ────────────────────────────────────────────────────────────
question = 'What is an embedding? Explain it in 3 sentences.'

print(f'→ Sending: "{question}"')
print('─' * 60)

reply = chat([{'role': 'user', 'content': question}])
pp(reply)

# ── Inspect the raw response ──────────────────────────────────────────────
print('\n── Raw response (selected fields) ──')
raw = requests.post(OLLAMA_URL, json={
    'model': MODEL, 'messages': [{'role': 'user', 'content': 'Hi'}], 'stream': False
}, timeout=30).json()
print(f"  role            : {raw['message']['role']}")
print(f"  done            : {raw['done']}")
ms = raw.get('total_duration', 0) / 1e6
print(f"  total_duration  : {ms:.0f} ms")

---

## Section 2 — Prompt Engineering Basics

### How prompt structure changes output quality

The same underlying model can produce wildly different outputs depending on
how you phrase the prompt. This is not magic — the model is completing a
sequence of tokens, so the tokens you provide at the start strongly constrain
what comes next.

Four patterns every engineer should know:

| Pattern | When to use | Typical improvement |
|---------|------------|---------------------|
| **Direct** | Baseline, simple questions | — |
| **System persona** | Tone, domain, format control | Better focus, fewer hallucinations |
| **Few-shot** | Classification, structured output | ±20–40% accuracy on structured tasks |
| **Chain-of-thought (CoT)** | Reasoning, maths, multi-step logic | Large gains on logic tasks |

### The role of system prompts

The `system` role message sets the *context frame* for the entire conversation.
It is processed before any user message and shapes every subsequent response.
Think of it as the job description you give to the model before the interview.

Key rules:
- Be specific about **role** (`You are a senior data engineer…`)
- Specify **output format** (`Reply with a single JSON object, nothing else`)
- Set **constraints** (`Do not mention competitor products`)

### What to look for
- The **direct** prompt gives a generic answer
- The **persona** prompt adds domain specificity and changes tone
- The **few-shot** prompt forces the exact output format you want
- The **CoT** prompt shows its reasoning before giving the final answer

In [ ]:
TOPIC = 'What is an embedding? Give a clear, concise answer.'

# ── Style 1: Direct ────────────────────────────────────────────────────────
print('=' * 70)
print('STYLE 1 — Direct prompt')
print('=' * 70)
reply_direct = chat([{'role': 'user', 'content': TOPIC}])
pp(reply_direct)

# ── Style 2: System persona ────────────────────────────────────────────────
print('\n' + '=' * 70)
print('STYLE 2 — System persona')
print('=' * 70)
reply_persona = chat([
    {
        'role': 'system',
        'content': (
            'You are a senior AI engineer teaching machine learning concepts to '
            'a data analyst who has never studied NLP. '
            'Use analogies and avoid mathematical notation. '
            'Keep your answer under 100 words.'
        ),
    },
    {'role': 'user', 'content': TOPIC},
])
pp(reply_persona)

# ── Style 3: Few-shot ──────────────────────────────────────────────────────
print('\n' + '=' * 70)
print('STYLE 3 — Few-shot (demonstrate the exact format you want)')
print('=' * 70)
few_shot_prompt = (
    'Define each AI term in exactly one sentence.\n\n'
    'Term: Token\n'
    'Definition: A token is a chunk of text (roughly 4 characters) that '
    'a language model processes as a single unit.\n\n'
    'Term: Transformer\n'
    'Definition: A transformer is a neural network architecture that uses '
    'self-attention to model relationships between all tokens in a sequence simultaneously.\n\n'
    'Term: Embedding\n'
    'Definition:'
)
reply_fewshot = chat([{'role': 'user', 'content': few_shot_prompt}], temperature=0.2)
pp(reply_fewshot)

# ── Style 4: Chain-of-thought ──────────────────────────────────────────────
print('\n' + '=' * 70)
print('STYLE 4 — Chain-of-thought')
print('=' * 70)
cot_prompt = (
    'Explain what an embedding is.\n'
    "Think step by step: first explain what the problem is that embeddings solve, "
    "then explain the solution, then give a one-sentence summary."
)
reply_cot = chat([{'role': 'user', 'content': cot_prompt}])
pp(reply_cot)

---

## Section 3 — Few-Shot Prompting for Classification

### Why examples in the prompt dramatically improve structured output

Zero-shot classification (`classify this ticket as high/medium/low`) works
roughly 60–70% of the time. Few-shot prompting — providing 3–5 *examples of
exactly the format you want* — pushes this to 85–95% without any fine-tuning.

Why? The model is a token predictor. When you show it three examples where the
output is the single word `high`, `medium`, or `low` with no surrounding text,
the token distribution for the next output is overwhelmingly concentrated on
those three tokens. The model has effectively been told the rules *and* shown
what correct answers look like.

### The recipe

1. Pick 3–5 diverse, unambiguous examples (one per class)
2. Use a consistent delimiter format (`Ticket: ... Priority:`)
3. Lower the temperature (`0.0`–`0.2`) to reduce creativity
4. Tell the model the vocabulary: `Reply with ONLY one word: high, medium, or low`

### What to look for
- The model should output a single word per ticket (no explanations)
- Compare predicted vs actual priority in the final table
- Where it goes wrong, look at the ticket text — is the true priority ambiguous?

In [ ]:
import os
import random
import pandas as pd

NOTEBOOK_DIR = os.path.abspath('')
DATA_DIR     = os.path.join(NOTEBOOK_DIR, '..', '..', 'data', 'raw')
TICKETS_PATH = os.path.join(DATA_DIR, 'support_tickets.csv')

# ── Auto-generate support_tickets.csv if missing ──────────────────────────
if not os.path.exists(TICKETS_PATH):
    print(f'support_tickets.csv not found — generating with faker...')
    try:
        from faker import Faker
        fake = Faker()
    except ImportError:
        raise SystemExit('Run: pip install faker')

    HIGH_TICKETS = [
        ('Production database is down', 'Our entire production database cluster has been unreachable for 30 minutes. All customers are affected. Revenue impact is severe.'),
        ('Critical security breach detected', 'We have detected unauthorised access to customer PII. Attack is ongoing. Immediate escalation required.'),
        ('Payment processing completely broken', 'No payments are going through. Checkout is broken for 100% of users. We are losing thousands per minute.'),
        ('API rate limits not enforced — data leak risk', 'Our public API has stopped enforcing rate limits. Malicious actors are scraping all customer data.'),
        ('All backups missing from last 7 days', 'Scheduled backup job silently failed for a week. We have no recovery point. Risk of permanent data loss.'),
        ('SSL certificate expired — site unreachable', 'Our SSL cert expired 2 hours ago. All users see a security warning and cannot access the platform.'),
        ('Authentication service returning 500 errors', 'Users cannot log in. Auth service throwing 500s since the last deployment 45 minutes ago.'),
        ('Data corruption in billing records', 'Invoice amounts are wrong for approximately 2,000 customers. Billing cycle runs in 6 hours.'),
    ]
    MEDIUM_TICKETS = [
        ('Dashboard export generates wrong totals', 'The CSV export from the analytics dashboard shows figures that do not match the on-screen numbers.'),
        ('Email notifications delayed by 4+ hours', 'Users report that welcome and password-reset emails arrive hours late, causing support calls.'),
        ('Search returns irrelevant results after update', 'Since last Monday\'s release, the search bar returns unrelated items frequently. Users are complaining.'),
        ('Mobile app crashes on iOS 17.4', 'Several users report the iOS app crashes immediately on launch after upgrading to iOS 17.4.'),
        ('Charts not rendering in Firefox', 'All bar and line charts are blank in Firefox 124+. Charts render correctly in Chrome and Safari.'),
        ('Bulk import fails on files larger than 5 MB', 'The CSV import wizard silently fails when the file exceeds 5 MB without showing an error to the user.'),
        ('Report generation taking over 10 minutes', 'Monthly summary reports that used to generate in 30 seconds are now taking 10+ minutes.'),
        ('Wrong timezone applied to scheduled tasks', 'Scheduled tasks appear to run in UTC rather than the user\'s configured timezone.'),
    ]
    LOW_TICKETS = [
        ('Request to add dark mode', 'Many users have requested a dark mode theme option. Would be nice to have for evening use.'),
        ('Typo on pricing page', 'The word \'anually\' should be \'annually\' on the /pricing page under the Pro plan.'),
        ('Tooltip text is hard to read', 'The tooltip on the settings page uses light grey text on white background. Hard to read.'),
        ('Add keyboard shortcut for save', 'It would be helpful to have Ctrl+S / Cmd+S save the current form automatically.'),
        ('Docs link broken on help page', 'The \'Getting Started\' link in the Help section returns a 404.'),
        ('Export button should confirm before running', 'Users accidentally trigger large exports. A confirmation dialog before exporting would prevent this.'),
        ('Date picker does not support manual entry', 'The date picker widget requires clicking; users who prefer typing cannot enter dates directly.'),
        ('Add option to change default currency', 'The UI always defaults to USD. Allow users to set a default display currency in their profile settings.'),
    ]

    random.seed(42)
    rows = []
    ticket_id = 1001
    for subject, body in HIGH_TICKETS * 5:
        rows.append({'id': ticket_id, 'subject': subject, 'body': body,
                     'priority': 'high', 'created_by': fake.name()})
        ticket_id += 1
    for subject, body in MEDIUM_TICKETS * 5:
        rows.append({'id': ticket_id, 'subject': subject, 'body': body,
                     'priority': 'medium', 'created_by': fake.name()})
        ticket_id += 1
    for subject, body in LOW_TICKETS * 5:
        rows.append({'id': ticket_id, 'subject': subject, 'body': body,
                     'priority': 'low', 'created_by': fake.name()})
        ticket_id += 1

    random.shuffle(rows)
    os.makedirs(DATA_DIR, exist_ok=True)
    pd.DataFrame(rows).to_csv(TICKETS_PATH, index=False)
    print(f'  Saved {len(rows)} tickets to {TICKETS_PATH}')

# ── Load tickets ──────────────────────────────────────────────────────────
tickets = pd.read_csv(TICKETS_PATH)
print(f'Loaded {len(tickets)} tickets  |  priority counts:')
print(tickets['priority'].value_counts().to_dict())

# ── Few-shot prompt builder ────────────────────────────────────────────────
EXAMPLES = [
    ('Production database is down — Our entire production database has been '
     'unreachable for 30 minutes. All customers affected.',
     'high'),
    ('Dashboard export generates wrong totals — The CSV export shows figures '
     'that do not match the on-screen numbers.',
     'medium'),
    ('Typo on pricing page — The word anually should be annually.',
     'low'),
]

def build_classification_prompt(subject: str, body: str) -> str:
    lines = [
        'Classify the priority of a support ticket as high, medium, or low.',
        'Reply with ONLY one word: high, medium, or low. No other text.\n',
    ]
    for ex_text, ex_label in EXAMPLES:
        lines.append(f'Ticket: {ex_text}')
        lines.append(f'Priority: {ex_label}\n')
    lines.append(f'Ticket: {subject} — {body}')
    lines.append('Priority:')
    return '\n'.join(lines)

# ── Run on 10 new tickets (not in the examples) ────────────────────────────
test_tickets = tickets.sample(10, random_state=99).reset_index(drop=True)

results = []
for _, row in test_tickets.iterrows():
    prompt   = build_classification_prompt(row['subject'], row['body'])
    predicted = chat([{'role': 'user', 'content': prompt}],
                     temperature=0.0).strip().lower()
    # Normalise — model might add punctuation
    predicted = predicted.split()[0].rstrip('.,;:') if predicted else 'unknown'
    correct   = predicted == row['priority']
    results.append({
        'subject':   row['subject'][:50],
        'actual':    row['priority'],
        'predicted': predicted,
        'correct':   '✓' if correct else '✗',
    })
    print(f"  [{row['priority']:6} → {predicted:6}] {'✓' if correct else '✗'}  {row['subject'][:45]}")

results_df = pd.DataFrame(results)
accuracy   = (results_df['correct'] == '✓').mean()
print(f'\nAccuracy: {accuracy:.0%}  ({int(accuracy*10)}/10 correct)')
display(results_df)

---

## Section 4 — Structured Output (JSON from an LLM)

### Why structured output matters for building systems

A language model outputs free text. To use that output programmatically —
store it in a database, display it in a UI, feed it into another system —
you need **structured output**: JSON, XML, CSV.

The challenge: LLMs sometimes add prose before the JSON, add trailing commas,
use single quotes instead of double, or hallucinate extra fields. You need
strategies to handle this:

1. **Instruction precision** — `Reply with ONLY valid JSON, no prose, no markdown`
2. **Low temperature** — reduces creativity and off-format outputs (`0.0`–`0.2`)
3. **Extraction fallback** — regex to find the `{...}` block if the model wraps it
4. **Schema enforcement** — newer Ollama versions support `format: "json"` in the payload

### The `format: "json"` shortcut

Ollama ≥ 0.1.9 supports `"format": "json"` in the request body, which
forces the model to output valid JSON regardless of the prompt. We use both
approaches below — explicit instruction + format key — for maximum reliability.

### What to look for
- `json.loads()` should succeed without errors
- The DataFrame should have columns: `subject`, `sentiment`, `suggested_action`
- Look at whether the `sentiment` label makes sense given the ticket body

In [ ]:
import re

SCHEMA_INSTRUCTION = '''You are a support triage assistant.
For the given ticket, return a JSON object with exactly these keys:
  - "subject":          one-sentence summary of the issue (max 12 words)
  - "sentiment":        one of: frustrated, neutral, urgent, polite
  - "suggested_action": one concrete action for the support agent (max 15 words)

Reply with ONLY the raw JSON object. No markdown. No explanations.'''


def extract_json(text: str) -> dict:
    """Parse JSON from model output; fallback to regex extraction."""
    text = text.strip()
    # Strip markdown code fences if present
    text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s*```$', '', text)
    # Try direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # Regex fallback: grab the first {...} block
    match = re.search(r'\{[^{}]+\}', text, re.DOTALL)
    if match:
        return json.loads(match.group())
    return {'subject': text[:60], 'sentiment': 'unknown', 'suggested_action': 'manual review'}


def summarise_ticket(row: pd.Series) -> dict:
    content = f'Ticket subject: {row["subject"]}\nTicket body: {row["body"]}'
    reply = chat(
        [
            {'role': 'system', 'content': SCHEMA_INSTRUCTION},
            {'role': 'user',   'content': content},
        ],
        temperature=0.1,
    )
    result = extract_json(reply)
    result['id']       = row['id']
    result['priority'] = row['priority']
    return result


# ── Process 8 tickets from different priority levels ─────────────────────
sample = (
    pd.concat([
        tickets[tickets['priority'] == 'high'].sample(3, random_state=1),
        tickets[tickets['priority'] == 'medium'].sample(3, random_state=1),
        tickets[tickets['priority'] == 'low'].sample(2, random_state=1),
    ])
    .reset_index(drop=True)
)

print(f'Summarising {len(sample)} tickets...\n')
summaries = []
for _, row in sample.iterrows():
    s = summarise_ticket(row)
    summaries.append(s)
    print(f"  id={s['id']}  priority={s['priority']:6}  sentiment={s.get('sentiment','?'):12}  "
          f"{s.get('subject','?')[:45]}")

summary_df = pd.DataFrame(summaries)[['id', 'priority', 'subject', 'sentiment', 'suggested_action']]
print()
display(summary_df)

---

## Section 5 — Building a Simple Chatbot Loop

### How message history works

A language model has no memory between API calls. The *illusion* of memory in
a chatbot is created by resending the **entire conversation history** with
every new request:

```
Turn 1:  [{user: "What is RAG?"}]
Turn 2:  [{user: "What is RAG?"}, {assistant: "RAG is..."}, {user: "Give an example"}]
Turn 3:  [{user: ...}, {assistant: ...}, {user: ...}, {assistant: ...}, {user: "Now compare to fine-tuning"}]
```

Each turn the `messages` list grows by two entries (one user, one assistant).
The model can therefore reference anything said earlier in the thread.

### Why context window matters

Each token in the history consumes context window capacity. `llama3` (8B) has
a context window of **8,192 tokens** (~6,000 words). Long conversations will
eventually exceed this and the oldest messages will be truncated — the model
will appear to "forget" the beginning of the chat.

Strategies to manage this in production:
- **Sliding window** — keep only the last N turns
- **Summarisation** — periodically compress history into a summary
- **RAG** — store conversation topics externally and retrieve relevant context (Phase 03!)

### Running this cell

The cell runs a chat loop in the notebook. Type your messages and press Enter.
Type **`quit`** or **`exit`** to end the session.

**Suggested follow-up questions to try:**
1. `What is an embedding?`
2. `How is that different from a keyword search?`
3. `Give me a concrete Python example`
4. `Now explain it as if I'm 10 years old`

In [ ]:
# ── Simple stateful chatbot ───────────────────────────────────────────────
# This cell is designed to run interactively in a Jupyter notebook.
# In VS Code / JupyterLab the input() prompt appears at the bottom of the cell.

SYSTEM_PROMPT = (
    'You are a helpful AI engineering tutor. '
    'You explain concepts clearly, use concrete examples, '
    'and always connect new ideas to what the student already knows. '
    'Keep responses concise — under 150 words unless asked for more detail.'
)

messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]

print('Chatbot started. Type "quit" or "exit" to end.')
print('─' * 60)

while True:
    user_input = input('You: ').strip()
    if not user_input:
        continue
    if user_input.lower() in ('quit', 'exit', 'q'):
        print('Goodbye! Conversation ended.')
        break

    messages.append({'role': 'user', 'content': user_input})

    reply = chat(messages)
    messages.append({'role': 'assistant', 'content': reply})

    print(f'\nAssistant: {reply}\n')
    print(f'[Context: {len(messages)-1} messages in history]')
    print('─' * 60)

# ── Print conversation transcript ─────────────────────────────────────────
print('\n── Conversation transcript ──')
for i, msg in enumerate(messages):
    if msg['role'] == 'system':
        continue
    role = 'You' if msg['role'] == 'user' else 'Assistant'
    print(f'\n[{i}] {role}:')
    pp(msg['content'])

---

## ✅ Summary — What You Learned

| # | Concept | Key takeaway |
|---|---------|-------------|
| 1 | Ollama REST API | `POST /api/chat` with a `messages` list returns a completion locally |
| 2 | Prompt engineering | System persona, few-shot, and CoT patterns improve output quality significantly |
| 3 | Few-shot classification | 3 examples in the prompt → 85–95% classification accuracy without fine-tuning |
| 4 | Structured output | Low temperature + JSON instruction + extraction fallback = reliable JSON parsing |
| 5 | Chatbot state | Append each turn to `messages[]` and resend the full list — that IS the memory |

### Key numbers to remember

- `llama3` context window: **8,192 tokens** (~6,000 words)
- Temperature `0.0` = deterministic output (best for structured tasks)
- Temperature `0.7`–`1.0` = creative output (best for brainstorming)
- Few-shot with 3 labelled examples typically closes 80% of the gap to fine-tuning

### What comes next — Phase 03: RAG Pipeline

You now know two things independently:

- **Phase 01** — how to turn text into vectors and search them semantically
- **Phase 02** — how to prompt a local LLM and get structured output

**Phase 03 combines them into a RAG (Retrieval-Augmented Generation) pipeline:**

```
User query
    │
    ▼
Embed query  ──▶  Search ChromaDB  ──▶  Top-k documents
                                              │
                                              ▼
                               Build prompt with context
                                              │
                                              ▼
                                    Ollama generates answer
                                              │
                                              ▼
                                    Grounded, cited response
```

The embedding search replaces the hallucination-prone knowledge in the LLM's
weights with *retrieved facts from your own documents*. This is the
architecture behind every enterprise AI assistant.